# Diversity clustering — US industry workforce composition

StrataScratch take-home. The file is anonymized summary data published with Google's Diversity Annual Report. Each row is a sector, subsector, industry group, or industry in a year, with a headcount and a demographic mix. We are clustering compositions, not people.

Questions split into two jobs. Only 1 and 6 are clustering.

1. Which sectors have a similar gender and ethnic structure?
2. Which three sectors are most gender-imbalanced, and how does that relate to size?
3. Which three sectors shifted most between 2021 and 2023?
4. Which sectors are dominated by one ethnic group?
5. Which sectors have the highest ethnic diversity, and how does that relate to size?
6. Can industries be grouped by ethnic composition?

Source notes: K-Means, hierarchical clustering, and DBSCAN in `yash9614/Nrish_Karik_SataDience` under `ML/`. Those notes are supervised-metric heavy (accuracy, MSE). Here there is no label, so the checks are silhouette, cluster profiles, and a PCA plot.


## 0. Setup

Percent columns are proportions, not 0–100. Hispanic/Latino is ethnicity and overlaps race, so the four race/ethnicity shares are not a closed pie. Employment is a weight for interpretation, not a clustering feature. Clustering on size would group large sectors together regardless of mix.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

FEATURES = [
    "percent_women",
    "percent_white",
    "percent_black_or_african_american",
    "percent_asian",
    "percent_hispanic_or_latino",
]
RACE_COLS = FEATURES[1:]


## 1. Load and grain

1,272 rows, 11 columns. `subsector` is null on the sector totals. `industry` is filled on only 507 rows, so the file mixes grains. One row is empty. Years are 2020–2023. Fourteen sectors, each present once a year, plus the overall line `Total, 16 years and over`.


In [ ]:
from pathlib import Path

candidates = [
    Path("/kaggle/input/datasets/paprikaryash/clustering-stratrascratch/dataset-google.csv"),
    Path("dataset-google.csv"),
    Path("../dataset-google.csv"),
]
path = next(p for p in candidates if p.exists())
df = pd.read_csv(path)
print(path)
print(df.shape)
df.info()
df.head(3)


In [ ]:
print("years:", sorted(df["year"].dropna().unique()))
print("empty rows:", len(df) - df.dropna(how="all").shape[0])
sector_rows = df[df["subsector"].isna() & df["sector"].notna()]
print("sector-level rows:", len(sector_rows))
print(sector_rows["sector"].value_counts())
print((df[RACE_COLS].sum(axis=1)).describe())


The race-share sum has a median above 1. Hispanic overlaps white, Black, and Asian, so we do not renormalize. The max of 129.9 is a scale error, not a composition.


## 2. Cleaning

Two rows were stored on a 0–100 scale (Construction 2023, nonmetallic mineral mining 2021). Divide those rows by 100. The 2021 overall Asian share is 0.660 against 0.06–0.07 in the neighboring years: one cell, divide by 10.

Asian shares above 0.25 elsewhere are real. Nail salons sit near 0.40 across years. Computer and peripheral equipment sits near 0.30. Internet publishing sits near 0.25. Do not recode a repeated industry pattern.


In [ ]:
work = df.dropna(how="all").copy()
scale_error = (work[FEATURES] > 1).any(axis=1)
clean = work.copy()
clean.loc[scale_error, FEATURES] = clean.loc[scale_error, FEATURES] / 100

total_2021 = (clean["sector"] == "Total, 16 years and over") & (clean["year"] == 2021)
clean.loc[total_2021, "percent_asian"] = clean.loc[total_2021, "percent_asian"] / 10
clean = clean.dropna(subset=["sector", "year"])
clean["year"] = clean["year"].astype(int)

print("rows rescaled:", int(scale_error.sum()))
print("shares still above 1:", int((clean[FEATURES] > 1).any(axis=1).sum()))
print(clean["percent_asian"].describe().round(3))

sector = clean[clean["subsector"].isna() & (clean["sector"] != "Total, 16 years and over")].copy()


## 3. Question 2 — gender imbalance

Imbalance is distance from 50% women. A sector at 15% women and a sector at 85% women are both imbalanced. Ranking by lowest female share would hide education and health services.

2023, sector grain. Construction 10.8% women (11.9 million), mining 15.3% (0.59 million), transportation 24.3% (9.9 million). Education and health is fourth, at 74.4% women and 36.4 million employed. The top three cover 22.4 million of 161 million. Mining is extreme and small. Construction and transportation are the workforce-scale gaps.


In [ ]:
s2023 = sector[sector["year"] == 2023].copy()
s2023["gender_gap"] = (s2023["percent_women"] - 0.5).abs()
s2023 = s2023.sort_values("gender_gap", ascending=False)
print(s2023[["sector", "total_employed_in_thousands", "percent_women", "gender_gap"]].to_string(index=False))
print("top 3 employed:", s2023["total_employed_in_thousands"].head(3).sum().round(0))
print("all sectors:", s2023["total_employed_in_thousands"].sum().round(0))


## 4. Question 3 — shift from 2021 to 2023

Shift is the sum of absolute changes across the five share columns, in percentage points. A sector can move without its women's share moving.

Largest shifts: mining (Hispanic +5.4 pp, 590 thousand employed — noisy), financial activities (white −1.9, Asian +1.2, 11.0 million), professional and business services (white −1.7, Black +1.1, 20.7 million). No sector transforms. The common move is about a 1 point decline in white share.


In [ ]:
keep = ["sector", *FEATURES, "total_employed_in_thousands"]
a = sector.loc[sector["year"] == 2021, keep].set_index("sector")
b = sector.loc[sector["year"] == 2023, keep].set_index("sector")
delta = b[FEATURES] - a[FEATURES]
delta["abs_shift"] = delta.abs().sum(axis=1)
delta["employed_2023"] = b["total_employed_in_thousands"]
delta = delta.sort_values("abs_shift", ascending=False)
show = delta.copy()
show[FEATURES + ["abs_shift"]] = show[FEATURES + ["abs_shift"]] * 100
print(show.round(2).to_string())


## 5. Question 4 — single-group dominance

White share is the max in every sector. That is the US workforce, not a model result. Extreme cases: agriculture 92.6%, mining 87.7%, construction 87.5%. Least concentrated: transportation 70.1%, public administration 72.2%. Construction is the one that is both extreme and large.

The second share is often Hispanic/Latino. That column overlaps race, so a 34% Hispanic share in construction is not a disjoint third of the workforce.


In [ ]:
s = sector[sector["year"] == 2023].set_index("sector").copy()
s["dominant_group"] = s[RACE_COLS].idxmax(axis=1)
s["dominant_share"] = s[RACE_COLS].max(axis=1)
s["second_share"] = s[RACE_COLS].apply(lambda r: r.nlargest(2).iloc[1], axis=1)
s["lead_pp"] = (s["dominant_share"] - s["second_share"]) * 100
out = s.sort_values("dominant_share", ascending=False)
print(
    out.assign(
        dominant_share=lambda d: (d["dominant_share"] * 100).round(1),
        second_share=lambda d: (d["second_share"] * 100).round(1),
        lead_pp=lambda d: d["lead_pp"].round(1),
    )[["dominant_group", "dominant_share", "second_share", "lead_pp", "total_employed_in_thousands"]].to_string()
)


## 6. Question 5 — ethnic diversity vs size

Simpson diversity on a closed race mix: white, Black, Asian, and the remainder as other. Zero if one group is everyone.

Highest: transportation 0.461 (Black share 20.7%), public administration 0.440, leisure 0.421, education and health 0.420. Lowest: agriculture 0.140, mining 0.225, construction 0.228. Size does not track diversity. Education and health is the largest sector and near the top. Construction is large and near the bottom.


In [ ]:
s["percent_other_race"] = (
    1 - s["percent_white"] - s["percent_black_or_african_american"] - s["percent_asian"]
).clip(lower=0)
race = ["percent_white", "percent_black_or_african_american", "percent_asian", "percent_other_race"]
s["simpson"] = 1 - (s[race] ** 2).sum(axis=1)
print(
    s.sort_values("simpson", ascending=False)[
        ["simpson", "percent_white", "percent_black_or_african_american", "percent_asian", "percent_other_race", "total_employed_in_thousands"]
    ]
    .assign(
        simpson=lambda d: d["simpson"].round(3),
        percent_white=lambda d: (d["percent_white"] * 100).round(1),
        percent_black_or_african_american=lambda d: (d["percent_black_or_african_american"] * 100).round(1),
        percent_asian=lambda d: (d["percent_asian"] * 100).round(1),
        percent_other_race=lambda d: (d["percent_other_race"] * 100).round(1),
    )
    .to_string()
)


## 7. Question 1 — sector clusters

K-Means minimizes within-cluster sum of squares. It needs a chosen k, and it is sensitive to scale. Women's share runs from 0.11 to 0.74. Asian share runs from 0.01 to 0.11. Without StandardScaler, gender would dominate the Euclidean distance and the fit would collapse into male vs female. Same reason the KNN notes scale features before a distance model.

Thirteen points. Silhouette is the check, because there is no label and accuracy does not apply. Inertia falls smoothly. Silhouette peaks at k = 2 (0.544) and drops after that. Taking k = 7 because inertia is smaller would be the unsupervised version of overfitting a flexible model.


In [ ]:
X_sector = sector.loc[sector["year"] == 2023, FEATURES].copy()
X_sector.index = sector.loc[sector["year"] == 2023, "sector"]
Z = StandardScaler().fit_transform(X_sector)

for k in range(2, 8):
    labels_k = KMeans(n_clusters=k, n_init=20, random_state=42).fit_predict(Z)
    km = KMeans(n_clusters=k, n_init=20, random_state=42).fit(Z)
    print(f"k={k}  inertia={km.inertia_:7.2f}  silhouette={silhouette_score(Z, labels_k):.3f}")


k = 2 is the sector solution. Cluster 1 is construction, agriculture, and mining: about 18% women, 89% white, 5% Black, 2% Asian, 27% Hispanic. 14.8 million employed, almost all of it construction. Cluster 0 is the other ten sectors, 146 million employed.

This is not a gender split. Education and health (74% women) stays in the majority cluster. Transportation (24% women) stays there too, because its Black share is 20.7%. The separator is the manual-outdoor mix: very white, very low Asian, low Black, elevated Hispanic.

Hierarchical clustering would be a reasonable alternative on 13 points (the notes build a dendrogram and cut it). With a silhouette this clean, the cut would land in the same place. DBSCAN is a poor fit: 13 points, no dense region versus noise, and epsilon would be a guess.


In [ ]:
labels = pd.Series(KMeans(n_clusters=2, n_init=20, random_state=42).fit_predict(Z), index=X_sector.index, name="cluster")
profile = X_sector.copy()
profile["cluster"] = labels
profile["employed"] = sector.loc[sector["year"] == 2023].set_index("sector")["total_employed_in_thousands"]
print(profile.sort_values("cluster").round(3).to_string())
print()
print(profile.groupby("cluster")[FEATURES].mean().round(3))
print(profile.groupby("cluster")["employed"].sum().round(0))


## 8. Question 6 — industry clusters

Industry grain, 2023, industry name filled. 127 industries. 46 employ fewer than 100,000. A percentage on a knitting mill of 7,000 people is a handful of survey respondents. Floor at 100,000 leaves 81 industries.

Silhouette is flat: 0.272 at k = 2, about 0.305 from k = 3 to k = 7. No tight industry types. Take k = 3, the smallest k on the plateau. A peak of 0.309 at k = 7 is not a reason to split further.


In [ ]:
ind = clean[(clean["year"] == 2023) & clean["industry"].notna()].copy()
print("industries:", len(ind), " under 100k:", int((ind["total_employed_in_thousands"] < 100).sum()))
ind = ind[ind["total_employed_in_thousands"] >= 100].copy()
X_ind = ind.set_index("industry")[FEATURES]
Z_ind = StandardScaler().fit_transform(X_ind)

for k in range(2, 9):
    lab = KMeans(n_clusters=k, n_init=20, random_state=42).fit_predict(Z_ind)
    print(f"k={k}  silhouette={silhouette_score(Z_ind, lab):.3f}")


In [ ]:
ind_labels = pd.Series(
    KMeans(n_clusters=3, n_init=20, random_state=42).fit_predict(Z_ind),
    index=X_ind.index,
    name="cluster",
)
ind_profile = X_ind.copy()
ind_profile["cluster"] = ind_labels
ind_profile["employed"] = ind.set_index("industry")["total_employed_in_thousands"]
print(ind_profile.groupby("cluster")[FEATURES].mean().round(3))
print(ind_profile.groupby("cluster").size().rename("n"))
print(ind_profile.groupby("cluster")["employed"].sum().round(0))
for c in sorted(ind_profile["cluster"].unique()):
    print(f"--- cluster {c} ---")
    print(ind_profile.loc[ind_profile["cluster"] == c, "employed"].nlargest(3).to_string())


Cluster 0 is three industries and 1.4 million employed: electronic components, computer equipment, nail salons. Mean Asian share 31.7%. This is the pocket we refused to recode.

Cluster 1 is service and care. Forty industries, 46.3 million in this filtered set. Restaurants, hospitals, outpatient care. 55.7% women, 17.8% Black, 22.0% Hispanic.

Cluster 2 is trades and property. Thirty-eight industries, 17.5 million. Real estate offices, landscaping, auto repair. 31.5% women, 84.2% white.

These employment totals are not sector totals. Rows without an industry name, and industries under 100,000, are excluded.


## 9. PCA check

PCA here is a plot, not a clustering method. Two components carry 43.5% and 26.2% (69.7%). Cluster 0 sits apart, low and left. Clusters 1 and 2 separate along PC1 and overlap near zero. That overlap is the silhouette of 0.30.


In [ ]:
pca = PCA(n_components=2, random_state=42)
xy = pca.fit_transform(Z_ind)
print("variance explained:", pca.explained_variance_ratio_.round(3))
print("loadings:")
print(pd.DataFrame(pca.components_, columns=FEATURES, index=["PC1", "PC2"]).round(3))

fig, ax = plt.subplots(figsize=(8, 5))
names = {0: "high Asian", 1: "service and care", 2: "trades and property"}
for c in sorted(ind_labels.unique()):
    m = ind_labels.to_numpy() == c
    ax.scatter(xy[m, 0], xy[m, 1], label=names[c], s=30)
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.legend()
ax.set_title("Industry clusters, 2023, employment >= 100k")
plt.tight_layout()
plt.show()


## Findings

- Data are BLS-style industry compositions, 2020–2023, not employee-level Google HR data.
- Two rows were on a 0–100 scale. One Asian cell on the 2021 total was a shifted decimal. High-Asian industries that repeat across years were left alone.
- Sector outlier group: construction, agriculture, mining. Construction is the policy-relevant one.
- Gender gaps that matter at scale: construction and transportation (male), education and health (female, just outside the top three on absolute gap).
- 2021–2023 shifts are a few points. Mining's Hispanic jump is on a small base.
- Every sector is white-dominated. Diversity does not rise with size.
- Industry fit is weak. One real pocket (high Asian) and two overlapping neighborhoods (service/care, trades/property).

What this uses from the ML notes, and what it does not: K-Means, scaling before a distance model, inertia versus a validation score, PCA as a view of the fit. Not used, on purpose: accuracy, RMSE, logistic regression, trees, boosting. Those need a label. DBSCAN and hierarchical clustering are the closer alternatives. DBSCAN does not suit 13 sector points. Hierarchical clustering would be the check to add if this were extended.
